# Sixpack_Chroma 벡터 재계산

보완 실험 계획에 따라, Sixpack_Chroma의 성능 저하 원인을 분석하기 위해
다양한 파라미터 조합으로 벡터를 재생성합니다.

## 실험 파라미터
| 파라미터 | 기존값 | 실험값 |
|---------|--------|--------|
| max_alpha | 10 | 10, 15, 20 |
| sigma (bandwidth) | 0.05 | 0.01, 0.02, 0.05, 0.1, 0.2 |
| H0 birth_range | (0, 0.01) | (0, 0.01), (0, 0.1), (0, 1.0) |

## 저장 경로
각 실험 조건별로 별도 디렉토리에 저장됩니다:
```
/content/drive/MyDrive/URP/1224_Vectors/Sixpack_Chroma_alpha{A}_sigma{S}_br{BR}/
```

## 1. 환경 설정

In [ ]:
from google.colab import drive

drive.mount('/content/drive/', force_remount=True)

In [ ]:
pip install chromatic_tda gudhi persim

In [ ]:
import chromatic_tda as chro
import matplotlib.pyplot as plt
import numpy as np
import os
import time
import json
from gudhi import RipsComplex
from gudhi.representations import PersistenceImage
from persim import PersistenceImager
import persim.images_weights as weights

print("All imports successful.")

## 2. 실험 설정

In [ ]:
# ============================================
# 실험 파라미터 설정
# ============================================

# 데이터 경로
BASE_DIR = "/content/drive/MyDrive/URP"
VECTOR_DIR = os.path.join(BASE_DIR, "1224_Vectors")

# 시뮬레이션 파라미터
A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1, x2, x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]
N_SAMPLES = len(PARAM_LIST)  # 512

# -------------------------------------------
# 실험할 파라미터 조합
# -------------------------------------------
EXPERIMENT_CONFIGS = [
    # (max_alpha, sigma, h0_birth_range_max, config_name)
    # 기존 설정 (baseline)
    (10,  0.05, 0.01, "baseline"),

    # sigma 변경 (max_alpha=10 고정)
    (10,  0.01, 0.01, "sigma0.01"),
    (10,  0.02, 0.01, "sigma0.02"),
    (10,  0.1,  0.01, "sigma0.1"),
    (10,  0.2,  0.01, "sigma0.2"),

    # max_alpha 변경 (sigma=0.05 고정)
    (15,  0.05, 0.01, "alpha15"),
    (20,  0.05, 0.01, "alpha20"),

    # H0 birth_range 확장 (max_alpha=10, sigma=0.05 고정)
    (10,  0.05, 0.1,  "br0.1"),
    (10,  0.05, 1.0,  "br1.0"),

    # 유망 조합: alpha 증가 + sigma 조정
    (15,  0.02, 0.01, "alpha15_sigma0.02"),
    (20,  0.02, 0.01, "alpha20_sigma0.02"),
    (15,  0.1,  0.01, "alpha15_sigma0.1"),
]

# 실행할 샘플 범위 (전체: 1~512, 테스트: 1~5 등)
START_IDX = 1
END_IDX = 512  # 전체 실행 시 512

# -------------------------------------------
# 실행할 실험 선택 (인덱스로)
# 전체 실행이 부담스러우면 일부만 선택 가능
# -------------------------------------------
SELECTED_EXPERIMENTS = list(range(len(EXPERIMENT_CONFIGS)))  # 전체 실행
# SELECTED_EXPERIMENTS = [0, 1, 2, 3, 4]  # 일부만 실행하려면 이렇게

print(f"총 {N_SAMPLES}개 샘플")
print(f"실행 범위: {START_IDX} ~ {END_IDX}")
print(f"\n실험 조건 목록 ({len(SELECTED_EXPERIMENTS)}개):")
for i in SELECTED_EXPERIMENTS:
    cfg = EXPERIMENT_CONFIGS[i]
    print(f"  [{i}] {cfg[3]}: max_alpha={cfg[0]}, sigma={cfg[1]}, h0_br_max={cfg[2]}")

## 3. 핵심 함수 정의

In [ ]:
def convert_into_diagram(diagram):
    """
    Converts kernel diagram dict → {dim: ndarray} and removes infinite deaths.
    """
    diagrams = {}
    for dim, pairs in diagram.items():
        filtered = [(float(b), float(d)) for (b, d) in pairs if np.isfinite(d)]
        diagrams[dim] = np.array(filtered)
    return diagrams


def convert_six_pack_to_diagram(six_pack):
    packs = {}
    for key, dgm in six_pack.items():
        pack = convert_into_diagram(dgm)
        packs[key] = pack
    return packs


def compute_six_pack_diagrams(points, labels, max_edge=10):
    """
    ChromaticAlphaComplex를 사용하여 six-pack barcode를 계산합니다.
    """
    chro_alpha = chro.ChromaticAlphaComplex(
        points, labels, max_alpha=max_edge
    )
    simplicial_complex = chro_alpha.get_simplicial_complex(
        sub_complex='0',
        full_complex='all',
        relative=None
    )
    six_pack = simplicial_complex.bars_six_pack()
    six_pack_dgms = convert_six_pack_to_diagram(six_pack)
    return six_pack_dgms


def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05,
                normalization=False, h0_birth_range_max=0.01):
    """
    Persistence Image 벡터를 계산합니다.

    Parameters:
        barcodes: dict, {dim: ndarray of (birth, death) pairs}
        max_eps: float, maximum epsilon (filtration value)
        px_res: float, pixel resolution
        sigma: float, Gaussian kernel bandwidth
        normalization: bool, 정규화 여부
        h0_birth_range_max: float, H0 birth range의 상한 (기존: 0.01)
    """
    # 키가 존재하지 않는 경우 빈 배열로 초기화
    if 0 not in barcodes:
        barcodes[0] = np.zeros((0, 2))
    if 1 not in barcodes:
        barcodes[1] = np.zeros((0, 2))

    for key in barcodes.keys():
        if len(barcodes[key]) == 0:
            barcodes[key] = np.zeros((0, 2))

    vector = {}

    # ========================
    # H0 Persistence Image
    # ========================
    pers_imager_h0 = PersistenceImager()
    pers_imager_h0.pixel_size = px_res
    pers_imager_h0.birth_range = (0, h0_birth_range_max)
    pers_imager_h0.pers_range = (0, max_eps)
    pers_imager_h0.weight = weights.persistence
    pers_imager_h0.weight_params = {'n': 1}
    pers_imager_h0.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h0 = np.array(barcodes.get(0, np.zeros((0, 2))))
    if len(bars_h0) > 0:
        img_h0 = pers_imager_h0.transform(bars_h0, skew=True)
    else:
        h0_birth_bins = max(1, int(h0_birth_range_max / px_res))
        h0_pers_bins = int(max_eps / px_res)
        img_h0 = np.zeros((h0_birth_bins, h0_pers_bins))

    # birth 축을 따라 평균 → 1D
    img0_1d = np.mean(img_h0, axis=0)

    # ========================
    # H1 Persistence Image
    # ========================
    pers_imager_h1 = PersistenceImager()
    pers_imager_h1.pixel_size = px_res
    pers_imager_h1.birth_range = (0, max_eps)
    pers_imager_h1.pers_range = (0, max_eps / 2)
    pers_imager_h1.weight = weights.persistence
    pers_imager_h1.weight_params = {'n': 1}
    pers_imager_h1.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h1 = np.array(barcodes.get(1, np.zeros((0, 2))))
    if len(bars_h1) > 0:
        img_h1 = pers_imager_h1.transform(bars_h1, skew=True)
    else:
        img_h1 = np.zeros((int(max_eps / px_res), int((max_eps / 2) / px_res)))

    # ========================
    # Normalization & Output
    # ========================
    if normalization:
        if np.max(img0_1d) > 0:
            vector[0] = img0_1d / np.max(img0_1d)
        else:
            vector[0] = img0_1d

        if np.max(img_h1) > 0:
            vector[1] = img_h1.flatten() / np.max(img_h1)
        else:
            vector[1] = img_h1.flatten()
    else:
        vector[0] = img0_1d
        vector[1] = img_h1.flatten()

    return vector


print("함수 정의 완료.")

## 4. 실험 실행 파이프라인

In [ ]:
def run_single_experiment(config_idx):
    """
    하나의 실험 조건에 대해 전체 샘플의 벡터를 생성합니다.
    """
    max_alpha, sigma, h0_br_max, config_name = EXPERIMENT_CONFIGS[config_idx]

    # 저장 디렉토리 생성
    out_dir = os.path.join(
        VECTOR_DIR,
        f"Sixpack_Chroma_alpha{max_alpha}_sigma{sigma}_br{h0_br_max}"
    )
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 70)
    print(f"실험: {config_name}")
    print(f"  max_alpha={max_alpha}, sigma={sigma}, h0_birth_range=(0, {h0_br_max})")
    print(f"  저장 경로: {out_dir}")
    print(f"  샘플 범위: {START_IDX} ~ {END_IDX}")
    print("=" * 70)

    total_start = time.time()
    success_count = 0
    error_count = 0
    skip_count = 0

    for idx in range(START_IDX, END_IDX + 1):
        save_path = os.path.join(out_dir, f"Sixpack_Chroma_{idx}.npz")

        # 이미 존재하면 건너뛰기
        if os.path.exists(save_path):
            skip_count += 1
            continue

        try:
            start_time = time.time()

            # 데이터 로드
            params = PARAM_LIST[idx - 1]
            folder = os.path.join(BASE_DIR, f"ParamSweep_{idx}_Output")
            pos_file = os.path.join(
                folder, f"Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat"
            )
            types_file = os.path.join(
                folder, f"Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat"
            )

            if not os.path.exists(pos_file) or not os.path.exists(types_file):
                print(f"  [{idx}] 데이터 파일 없음, 건너뜀")
                error_count += 1
                continue

            types = np.loadtxt(types_file, dtype=int)
            positions = np.loadtxt(pos_file, delimiter=',')

            A = positions[types == 1]
            B = positions[types == 2]
            points = np.array(np.concatenate([A, B], axis=0))
            labels = np.array(np.concatenate([np.zeros(len(A)), np.ones(len(B))]))

            # Six-pack barcode 계산 (양방향)
            six_pack_A_to_B = compute_six_pack_diagrams(
                points, labels, max_edge=max_alpha
            )
            six_pack_B_to_A = compute_six_pack_diagrams(
                points, 1 - labels, max_edge=max_alpha
            )

            # Persistence Image 벡터화
            PI_A_to_B = {}
            PI_B_to_A = {}

            for key in six_pack_A_to_B.keys():
                PI_A_to_B[key] = compute_PIs(
                    six_pack_A_to_B[key],
                    max_eps=max_alpha,
                    sigma=sigma,
                    h0_birth_range_max=h0_br_max,
                    normalization=False
                )

            for key in six_pack_B_to_A.keys():
                PI_B_to_A[key] = compute_PIs(
                    six_pack_B_to_A[key],
                    max_eps=max_alpha,
                    sigma=sigma,
                    h0_birth_range_max=h0_br_max,
                    normalization=False
                )

            # 저장
            np.savez_compressed(save_path, PI_A_to_B, PI_B_to_A)

            elapsed = time.time() - start_time
            success_count += 1

            if idx % 50 == 0 or idx == START_IDX:
                print(f"  [{idx}/{END_IDX}] 완료 ({elapsed:.2f}초)")

        except Exception as e:
            print(f"  [{idx}] 오류: {e}")
            error_count += 1

    total_elapsed = time.time() - total_start
    print(f"\n--- {config_name} 실험 완료 ---")
    print(f"  성공: {success_count}, 건너뜀: {skip_count}, 오류: {error_count}")
    print(f"  총 소요 시간: {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")
    print()

    return {
        'config_name': config_name,
        'max_alpha': max_alpha,
        'sigma': sigma,
        'h0_br_max': h0_br_max,
        'success': success_count,
        'skipped': skip_count,
        'errors': error_count,
        'total_time': total_elapsed
    }


print("파이프라인 함수 정의 완료.")

## 5. 테스트 실행 (소규모)

전체 실행 전에 먼저 소수의 샘플로 테스트합니다.

In [ ]:
# 테스트: 첫 3개 샘플만, baseline 설정으로
START_IDX = 1
END_IDX = 3

test_result = run_single_experiment(0)  # baseline
print("테스트 결과:", test_result)

## 6. 전체 실행

테스트가 정상적으로 완료되면 아래 셀을 실행하세요.

⚠️ **주의**: 전체 실행 시 실험 1개당 약 50분, 총 12개 실험 시 약 10시간 소요 예상.
필요한 실험만 선택적으로 실행하세요.

In [ ]:
# 전체 샘플 범위로 변경
START_IDX = 1
END_IDX = 512

# 선택적 실행: 필요한 실험 인덱스를 지정
# 전체: list(range(len(EXPERIMENT_CONFIGS)))
# 우선순위 높은 것만: [0, 1, 2, 3, 4]  (baseline + sigma 변경)
SELECTED_EXPERIMENTS = [0, 1, 2, 3, 4]  # sigma 변경 실험 우선 실행

all_results = []
for exp_idx in SELECTED_EXPERIMENTS:
    result = run_single_experiment(exp_idx)
    all_results.append(result)

# 결과 요약
print("\n" + "=" * 70)
print("전체 실험 결과 요약")
print("=" * 70)
for r in all_results:
    print(f"  {r['config_name']:25s} | 성공: {r['success']:3d} | "
          f"시간: {r['total_time']/60:.1f}분")

## 7. max_alpha 변경 실험 (barcode 재계산 필요)

max_alpha를 변경하면 barcode 자체가 달라지므로 별도로 실행합니다.

In [ ]:
# max_alpha 변경 실험
START_IDX = 1
END_IDX = 512

# alpha15, alpha20, alpha15_sigma0.02, alpha20_sigma0.02, alpha15_sigma0.1
ALPHA_EXPERIMENTS = [5, 6, 9, 10, 11]

alpha_results = []
for exp_idx in ALPHA_EXPERIMENTS:
    result = run_single_experiment(exp_idx)
    alpha_results.append(result)

print("\n" + "=" * 70)
print("max_alpha 변경 실험 결과 요약")
print("=" * 70)
for r in alpha_results:
    print(f"  {r['config_name']:25s} | 성공: {r['success']:3d} | "
          f"시간: {r['total_time']/60:.1f}분")

## 8. birth_range 변경 실험

In [ ]:
# H0 birth_range 변경 실험
START_IDX = 1
END_IDX = 512

BR_EXPERIMENTS = [7, 8]  # br0.1, br1.0

br_results = []
for exp_idx in BR_EXPERIMENTS:
    result = run_single_experiment(exp_idx)
    br_results.append(result)

print("\n" + "=" * 70)
print("birth_range 변경 실험 결과 요약")
print("=" * 70)
for r in br_results:
    print(f"  {r['config_name']:25s} | 성공: {r['success']:3d} | "
          f"시간: {r['total_time']/60:.1f}분")

## 9. 생성된 벡터 검증

각 실험 조건별로 생성된 벡터의 shape과 통계량을 확인합니다.

In [ ]:
import glob

print("생성된 벡터 디렉토리 스캔:")
print("=" * 70)

chroma_dirs = sorted(glob.glob(os.path.join(VECTOR_DIR, "Sixpack_Chroma_*")))

for d in chroma_dirs:
    dir_name = os.path.basename(d)
    npz_files = sorted(glob.glob(os.path.join(d, "Sixpack_Chroma_*.npz")))
    n_files = len(npz_files)

    if n_files == 0:
        print(f"  {dir_name}: 파일 없음")
        continue

    # 첫 번째 파일로 shape 확인
    try:
        sample = np.load(npz_files[0], allow_pickle=True)
        arr_0 = sample['arr_0'].item()
        arr_1 = sample['arr_1'].item()

        # 전체 벡터 크기 추정
        total_dim = 0
        detail_parts = []
        for direction_name, direction_data in [('A→B', arr_0), ('B→A', arr_1)]:
            for key in sorted(direction_data.keys()):
                for dim_key in sorted(direction_data[key].keys()):
                    vec = direction_data[key][dim_key]
                    if hasattr(vec, 'flatten'):
                        size = len(vec.flatten())
                    else:
                        size = len(np.array(vec).flatten())
                    total_dim += size

        print(f"  {dir_name}: {n_files}개 파일, 총 벡터 차원 ≈ {total_dim}D")
    except Exception as e:
        print(f"  {dir_name}: {n_files}개 파일, 검증 오류: {e}")

print("\n검증 완료.")